The goal of this file is to use two stage of extractive methods to create a summary of financial article
- First stage : DistilBert
- Second stage : LexRank

The idea is to make a summary smaller that DistilBert who is better at other method used but with summary little too long. 

In [24]:
import nltk
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score
from rouge_score import rouge_scorer
from summarizer import Summarizer

nltk.download('punkt')

nltk.download('reuters')
from nltk.tokenize import sent_tokenize
from nltk.corpus import reuters
from lexrank import LexRank
from lexrank.mappings.stopwords import STOPWORDS

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Justine\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\Justine\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!


- Retrieve the text in input (Financial Article)

In [2]:
file = "Text_Input.txt"

file_path = rf"D:\Project\Trading\Stage_L3\Extractive Method\{file}"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
#Compute METRICS ROUGE and BERTScore
def ComputeMetrics(summary,benchmark_summary,metric,flag):
    #summary generated
    print("--- Summary generated ---")
    if flag :
        summary = " ".join(summary)  
    print(summary)

    #benchmark summary
    print("--- benchmark summary ---")
    benchmark_summary = " ".join(benchmark_summary)  
    print(benchmark_summary)

    match metric:
        case 'bertscore':
            #Compute BERTScore
            P, R, F1 = score([summary], [benchmark_summary], lang="en", verbose=True)

            print("\n--- Scores BERTscore (compared to benchmark summary) ---")
            print(f"Precision: {P.mean():.4f}")
            print(f"Recall: {R.mean():.4f}")
            print(f"F1: {F1.mean():.4f}")

        case 'rouge':
            #Compute ROUGE
            scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
            scores = scorer.score(benchmark_summary, summary)

            print("\n--- Scores ROUGE (compared to benchmark summary) ---")
            for key, value in scores.items():
                print(f"{key}: Precision={value.precision:.3f} Recall={value.recall:.3f} ")

- First Stage (DistilBert)

In [11]:
model = Summarizer(model='distilbert-base-uncased')

summary_first = model(text, min_length=30)
print(summary_first)


Alphabet's (GOOGL.O), opens new tab Google said on Tuesday it would put artificial intelligence into the hands of more Web surfers while teasing a $249.99-a-month subscription for its AI power users, its latest effort to fend off growing competition from startups like OpenAI. Google unveiled the plans amid a flurry of demos that included new smart glasses during its annual I/O conference in Mountain View, California, which has adopted a tone of increased urgency since the rise of generative AI challenged the tech company's longtime stronghold of organizing and retrieving information on the internet. In a major update, the company said consumers across the United States now can switch Google Search into “AI Mode.” Google's new plan also includes 30 terabytes of cloud storage and an ad-free YouTube subscription. On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' lenses. In turn, some analys

In [12]:
benchmark_summary = ["Google brings 'AI Mode' to all U.S. users in bid to maintain search dominance. Analysts predict Google's search market share may drop below 50% in five years. Google touts glasses, new AI features as step towards 'universal' assistants"]
ComputeMetrics(summary_first,benchmark_summary,"bertscore",False)

--- Summary generated ---
Alphabet's (GOOGL.O), opens new tab Google said on Tuesday it would put artificial intelligence into the hands of more Web surfers while teasing a $249.99-a-month subscription for its AI power users, its latest effort to fend off growing competition from startups like OpenAI. Google unveiled the plans amid a flurry of demos that included new smart glasses during its annual I/O conference in Mountain View, California, which has adopted a tone of increased urgency since the rise of generative AI challenged the tech company's longtime stronghold of organizing and retrieving information on the internet. In a major update, the company said consumers across the United States now can switch Google Search into “AI Mode.” Google's new plan also includes 30 terabytes of cloud storage and an ad-free YouTube subscription. On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' le

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:18<00:00, 18.78s/it]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00,  6.70it/s]


done in 18.94 seconds, 0.05 sentences/sec

--- Scores BERTscore (compared to benchmark summary) ---
Precision: 0.8181
Recall: 0.8862
F1: 0.8508


In [13]:
ComputeMetrics(summary_first,benchmark_summary,"rouge",False)

--- Summary generated ---
Alphabet's (GOOGL.O), opens new tab Google said on Tuesday it would put artificial intelligence into the hands of more Web surfers while teasing a $249.99-a-month subscription for its AI power users, its latest effort to fend off growing competition from startups like OpenAI. Google unveiled the plans amid a flurry of demos that included new smart glasses during its annual I/O conference in Mountain View, California, which has adopted a tone of increased urgency since the rise of generative AI challenged the tech company's longtime stronghold of organizing and retrieving information on the internet. In a major update, the company said consumers across the United States now can switch Google Search into “AI Mode.” Google's new plan also includes 30 terabytes of cloud storage and an ad-free YouTube subscription. On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' le

- Second Stage (LexRank)

In [ ]:
documents = []
for fileid in reuters.fileids()[:1000]: 
    raw_text = reuters.raw(fileid)
    sentences = sent_tokenize(raw_text)
    documents.append(sentences)

lxr = LexRank(documents, stopwords=STOPWORDS['en'])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Justine\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\Justine\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!


In [27]:
sentences = sent_tokenize(summary_first)

#execute the model with our text
summary_second = lxr.get_summary(sentences, summary_size=3, threshold=0.2)


print("\nSummary:")
for line in summary_second:
    print("-", line)


Summary:
- In a number of demos, Google drew on capabilities developed in a testing ground it has called Project Astra to show off what its latest AI could do.
- In turn, some analysts reassessed how to measure Google's dominant search market share, with one estimate stating it could fall to less than 50% from around 90% in five years.
- On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' lenses.


In [28]:
ComputeMetrics(summary_second,benchmark_summary,"bertscore",True)

--- Summary generated ---
In a number of demos, Google drew on capabilities developed in a testing ground it has called Project Astra to show off what its latest AI could do. In turn, some analysts reassessed how to measure Google's dominant search market share, with one estimate stating it could fall to less than 50% from around 90% in five years. On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' lenses.
--- benchmark summary ---
Google brings 'AI Mode' to all U.S. users in bid to maintain search dominance. Analysts predict Google's search market share may drop below 50% in five years. Google touts glasses, new AI features as step towards 'universal' assistants


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:18<00:00, 18.57s/it]


computing greedy matching.


100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


done in 20.60 seconds, 0.05 sentences/sec

--- Scores BERTscore (compared to benchmark summary) ---
Precision: 0.8501
Recall: 0.8826
F1: 0.8661


In [29]:
ComputeMetrics(summary_second,benchmark_summary,"rouge",True)

--- Summary generated ---
In a number of demos, Google drew on capabilities developed in a testing ground it has called Project Astra to show off what its latest AI could do. In turn, some analysts reassessed how to measure Google's dominant search market share, with one estimate stating it could fall to less than 50% from around 90% in five years. On stage, two Google officials had a conversation in different languages while the glasses typed up translations for them, viewed through the frames' lenses.
--- benchmark summary ---
Google brings 'AI Mode' to all U.S. users in bid to maintain search dominance. Analysts predict Google's search market share may drop below 50% in five years. Google touts glasses, new AI features as step towards 'universal' assistants

--- Scores ROUGE (compared to benchmark summary) ---
rouge1: Precision=0.217 Recall=0.450 
rouge2: Precision=0.061 Recall=0.128 
rougeL: Precision=0.181 Recall=0.375 


In conclusion, the first stage of the summarization process serves to perform an initial filtering of sentences. This stage generates a denser summary by selecting the most relevant segments from the original text. However, despite its relevance, the resulting summary remains relatively long and may contain some redundancy.

The second stage aims to further refine the summary produced by DistilBERT by reducing its length while preserving its core meaning. This step improves the overall quality of the summary, as demonstrated by a higher BERTScore. In contrast, LexRank, although effective in some cases, tends to omit important information, which affects the informativeness of its output.

When comparing the summaries produced by each method to the reference benchmark summary, it is clear that LexRank produces the version that is closest in structure and content, but only after the DistilBERT stage has been applied. This suggests that combining both methods in a multi-stage summarization process can lead to more accurate and concise summaries.